## 1. Kurulum ve Bağımlılıklar

In [ ]:
# GPU kontrolü
!nvidia-smi

In [ ]:
# Repository'yi klonla
!git clone https://github.com/Aliekinozcetin/DiffuVQA.git
%cd DiffuVQA

# Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

# Dataset images'ı SMART COPY (sadece gerekli görseller!)
import os
import json
import shutil
from pathlib import Path
from tqdm import tqdm

def find_image_with_extensions(base_path, img_name):
    """Farklı uzantıları deneyen görüntü bulucu"""
    base, ext = os.path.splitext(img_name)
    extensions = [ext, '.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']
    
    for try_ext in extensions:
        full_path = os.path.join(base_path, base + try_ext)
        if os.path.exists(full_path):
            return full_path, base + try_ext
    return None, None

# SLAKE dataset: Sadece train/test/valid'de kullanılan görselleri kopyala
if not os.path.exists('./datasets/imgs'):
    print("📥 Gerekli görüntüler belirleniyor...")
    
    # JSONL dosyalarından kullanılan görüntü yollarını topla
    required_images = set()
    for jsonl_file in ['datasets/train.jsonl', 'datasets/valid.jsonl', 'datasets/test.jsonl']:
        if os.path.exists(jsonl_file):
            with open(jsonl_file, 'r') as f:
                for line in f:
                    data = json.loads(line)
                    img_path = data.get('img_name') or data.get('image_path') or data.get('image')
                    if img_path:
                        img_path = img_path.replace('imgs/', '')
                        required_images.add(img_path)
    
    print(f"📋 {len(required_images)} unique görüntü bulundu\n")
    
    # Google Drive'daki SLAKE dataset yolunu otomatik bul
    possible_paths = [
        "/content/drive/MyDrive/SLAKE_dataset/imgs",
        "/content/drive/Drive'ım/SLAKE_dataset/imgs",
        "/content/drive/MyDrive/SLAKE/imgs",
        "/content/drive/Drive'ım/SLAKE/imgs",
    ]
    
    source_dir = None
    for path in possible_paths:
        if os.path.exists(path):
            test_img = list(required_images)[0]
            found_path, _ = find_image_with_extensions(path, test_img)
            if found_path:
                source_dir = path
                print(f"✅ Dataset bulundu: {source_dir}")
                break
    
    if source_dir is None:
        print("❌ SLAKE dataset bulunamadı! Denenen yollar:")
        for path in possible_paths:
            print(f"   • {path}")
        raise FileNotFoundError("SLAKE dataset images folder not found")
    
    print("📥 Kopyalama başlıyor...\n")
    os.makedirs('./datasets/imgs', exist_ok=True)
    
    copied_count = 0
    failed_images = []
    
    for img in tqdm(required_images, desc="Kopyalanıyor"):
        src_path, actual_filename = find_image_with_extensions(source_dir, img)
        
        if src_path:
            try:
                dst = os.path.join("./datasets/imgs", img)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copy2(src_path, dst)
                copied_count += 1
            except Exception as e:
                failed_images.append((img, str(e)))
        else:
            failed_images.append((img, "Not found"))
    
    print(f"\n✅ {copied_count}/{len(required_images)} görüntü kopyalandı")
    
    if failed_images:
        print(f"⚠️  {len(failed_images)} görüntü kopyalanamadı:")
        for img, reason in failed_images[:5]:
            print(f"   • {img}: {reason}")
else:
    print("✅ Image klasörü zaten mevcut")

In [ ]:
# Dataset yapısını kontrol et
!echo "📂 Dataset klasörü:"
!ls -lh datasets/
!echo "\n📊 JSONL dosyaları:"
!ls -lh datasets/*.jsonl
!echo "\n🖼️ Image klasörü (symlink):"
!ls -lh datasets/imgs/ | head -5
!echo "\n📄 Train set örneği:"
!head -n 1 datasets/train.jsonl

In [ ]:
# Bağımlılıkları yükle
# Colab PyTorch ile geliyor, o yüzden requirements_colab.txt kullanıyoruz
!pip install --upgrade pip
!pip install -r requirements_colab.txt
!pip install pandas openpyxl bert-score  # Excel support ve BERTScore için
!python -m spacy download en_core_web_sm

print("\n✅ Kurulum tamamlandı!")

In [ ]:
# Gerekli kütüphaneleri içe aktar
import os
import sys
import json
import torch
import pandas as pd
import numpy as np
from datetime import datetime
from collections import defaultdict
import glob

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Model Eğitimi (PubMedBERT)

In [ ]:
# Eğitim konfigürasyonu
training_config = {
    "vocab": "pubmedbert",  # PubMedBERT tokenizer kullan
    "use_plm_init": "pubmedbert",  # PubMedBERT embeddings kullan (init_pretrained DEĞİL!)
    "batch_size": 4,
    "learning_rate": 0.0001,
    "learning_steps": 1000,  # num_steps DEĞİL, learning_steps!
    "diffusion_steps": 100,
    "seq_len": 128,
    "hidden_size": 768,  # PubMedBERT hidden dimension
    "image_resolution": 224,
    "seed": 42,
    "checkpoint_path": "./checkpoints/pubmedbert_slake"
}

# Checkpoint dizinini oluştur
os.makedirs(training_config["checkpoint_path"], exist_ok=True)

print("Eğitim Konfigürasyonu:")
for k, v in training_config.items():
    print(f"  {k}: {v}")

In [ ]:
# Model eğitimini başlat - T4 GPU optimized
# NOT: train.py parametreleri:
# - vocab: tokenizer seçimi (bert, pubmedbert)
# - use_plm_init: model initialization (bert, pubmedbert)
# - learning_steps: toplam eğitim adımı (num_steps DEĞİL!)
# - data_dir: dataset klasör yolu
# - dataset: dataset adı

!python train.py \
    --vocab pubmedbert \
    --use_plm_init pubmedbert \
    --batch_size 8 \
    --lr 0.0001 \
    --diffusion_steps 100 \
    --seq_len 128 \
    --hidden_t_dim 768 \
    --checkpoint_path ./checkpoints/pubmedbert_slake \
    --seed 42 \
    --learning_steps 4000 \
    --save_interval 200 \
    --log_interval 50 \
    --data_dir datasets \
    --dataset SLAKE \
    --image_dir datasets/imgs

## 3. Model Örnekleme (Inference)

In [ ]:
# Test seti üzerinde örnekleme yap
# Checkpoint dosyasını otomatik bul
import glob
checkpoint_dir = "./checkpoints/pubmedbert_slake/"
checkpoint_files = sorted(glob.glob(f"{checkpoint_dir}/ema_*.pt"))

if checkpoint_files:
    # Son checkpoint'i kullan
    checkpoint_file = checkpoint_files[-1]
    print(f"📂 Kullanılacak checkpoint: {checkpoint_file}")
else:
    print("⚠️ Checkpoint bulunamadı! Lütfen önce eğitimi tamamlayın.")
    checkpoint_file = None

if checkpoint_file:
    # sample_vqa_GPU.py config.json'dan parametreleri okuyor
    # T4 GPU ile batch_size=8, 50 steps (memory optimized)
    !python sample_vqa_GPU.py \
        --model_path {checkpoint_file} \
        --step 50 \
        --batch_size 8 \
        --seed2 105 \
        --split test \
        --clamp_step 0
    
    print(f"\n✅ Örnekleme tamamlandı!")
    print(f"📂 Çıktı dosyaları samples/ klasöründe oluşturuldu.")
    
    # Oluşturulan dosyaları listele
    !echo "\n📄 Oluşturulan sample dosyaları:"
    !ls -lh samples/*.jsonl | tail -5

## 4. Model Değerlendirme ve CSV Export

In [ ]:
# Değerlendirme fonksiyonları
import glob
import json
import pandas as pd
from torchmetrics.text.rouge import ROUGEScore
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk

# NLTK veri setlerini indir
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

rougeScore = ROUGEScore()

def get_bleu(recover, reference, n=1):
    """BLEU-n score"""
    weights = tuple((1.0 / n for _ in range(n)))
    return sentence_bleu([reference.split()], recover.split(), 
                        weights=weights, 
                        smoothing_function=SmoothingFunction().method4)

In [ ]:
def evaluate_and_export_csv(sample_files, output_csv="evaluation_results.csv", dataset_file="datasets/test.jsonl"):
    """
    Örneklenmiş model çıktılarını değerlendir ve CSV'ye kaydet
    
    Args:
        sample_files: JSONL formatında örnek dosyaları (liste veya tek dosya)
        output_csv: Çıktı CSV dosya yolu
        dataset_file: Original dataset (answer_type bilgisi için)
    """
    if isinstance(sample_files, str):
        sample_files = [sample_files]
    
    # Dataset'ten answer_type mapping'i oluştur
    print(f"📂 Dataset'ten answer_type bilgisi yükleniyor: {dataset_file}")
    answer_type_map = {}
    try:
        with open(dataset_file, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                # qid veya question ile eşleştir
                qid = data.get('qid')
                question = data.get('question', '').strip().lower()
                answer_type = data.get('answer_type', 'UNKNOWN')
                
                if qid:
                    answer_type_map[qid] = answer_type
                if question:
                    answer_type_map[question] = answer_type
        print(f"✅ {len(answer_type_map)} answer_type mapping yüklendi")
    except Exception as e:
        print(f"⚠️ Dataset yüklenemedi: {e}")
        answer_type_map = {}
    
    all_results = []
    
    for sample_file in sample_files:
        print(f"\n📊 Değerlendiriliyor: {sample_file}")
        
        # JSONL dosyasını oku
        samples = []
        with open(sample_file, 'r', encoding='utf-8') as f:
            for line in f:
                samples.append(json.loads(line))
        
        print(f"✅ {len(samples)} satır okundu")
        
        # Metrikler için listeler
        bleu1_scores = []
        rougeL_scores = []
        meteor_scores = []
        f1_scores = []
        
        predictions = []
        references = []
        
        # Accuracy için sayaçlar
        total_samples = 0
        correct_all = 0
        correct_yn = 0
        correct_oe = 0
        count_yn = 0
        count_oe = 0
        empty_count = 0
        
        # Her örnek için metrikleri hesapla
        for sample in samples:
            # Farklı JSON key'lerini dene
            pred = (sample.get('generate_answer') or 
                   sample.get('recover') or 
                   sample.get('generated_answer') or 
                   sample.get('prediction') or '').strip()
            
            ref = (sample.get('reference_answer') or 
                  sample.get('reference') or 
                  sample.get('answer') or '').strip()
            
            # answer_type'ı JSON'dan veya mapping'den al
            q_type = sample.get('answer_type', 'UNKNOWN')
            
            # Eğer sample'da yoksa, dataset'ten eşleştir
            if q_type == 'UNKNOWN':
                qid = sample.get('qid')
                question = sample.get('question', '').strip().lower()
                
                if qid and qid in answer_type_map:
                    q_type = answer_type_map[qid]
                elif question and question in answer_type_map:
                    q_type = answer_type_map[question]
            
            if not ref:
                continue
            
            # Boş cevapları say
            if not pred:
                empty_count += 1
                pred = "[EMPTY]"
            
            predictions.append(pred)
            references.append(ref)
            
            # Exact match için case-insensitive karşılaştırma
            em = 1.0 if pred.lower().strip() == ref.lower().strip() else 0.0
            
            # Accuracy hesapla
            total_samples += 1
            if em == 1.0:
                correct_all += 1
                if q_type == 'CLOSED':
                    correct_yn += 1
                elif q_type == 'OPEN':
                    correct_oe += 1
            
            if q_type == 'CLOSED':
                count_yn += 1
            elif q_type == 'OPEN':
                count_oe += 1
            
            # F1 Score (token-level)
            if pred != "[EMPTY]":
                pred_tokens = set(pred.lower().split())
                ref_tokens = set(ref.lower().split())
                if len(pred_tokens) == 0 or len(ref_tokens) == 0:
                    f1 = 0.0
                else:
                    precision = len(pred_tokens & ref_tokens) / len(pred_tokens)
                    recall = len(pred_tokens & ref_tokens) / len(ref_tokens)
                    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
                f1_scores.append(f1)
            else:
                f1_scores.append(0.0)
            
            # BLEU-1
            if pred != "[EMPTY]":
                bleu1_scores.append(get_bleu(pred, ref, n=1))
            else:
                bleu1_scores.append(0.0)
            
            # ROUGE-L
            if pred != "[EMPTY]":
                rougeL_scores.append(rouge_l_score(pred, ref))
            else:
                rougeL_scores.append(0.0)
            
            # METEOR
            if pred != "[EMPTY]":
                meteor_scores.append(calculate_meteor(pred, ref))
            else:
                meteor_scores.append(0.0)
        
        # CIDEr-like score
        cider = cider_score(predictions, references) if predictions else 0.0
        
        # BERTScore hesapla
        print("⏳ BERTScore hesaplanıyor...")
        try:
            from bert_score import score as bert_score_fn
            # Boş cevapları filtrele
            valid_preds = [p for p in predictions if p != "[EMPTY]"]
            valid_refs = [r for i, r in enumerate(references) if predictions[i] != "[EMPTY]"]
            
            if valid_preds and valid_refs:
                bert_precision, bert_recall, bert_f1 = bert_score_fn(
                    valid_preds, valid_refs, lang='en', verbose=False, device='cuda' if torch.cuda.is_available() else 'cpu'
                )
                bert_f1_score = bert_f1.mean().item()
            else:
                bert_f1_score = 0.0
        except Exception as e:
            print(f"⚠️ BERTScore hesaplanamadı: {e}")
            bert_f1_score = 0.0
        
        # Debug: Answer type dağılımını yazdır
        print(f"\n📋 Answer Type Dağılımı:")
        print(f"  CLOSED sorular: {count_yn}")
        print(f"  OPEN sorular: {count_oe}")
        print(f"  UNKNOWN: {total_samples - count_yn - count_oe}")
        
        # Sample folder path'i al
        sample_folder = os.path.dirname(sample_file) if os.path.dirname(sample_file) else "./samples"
        
        # SENİN İSTEDİĞİN SIRAYLA KOLONLAR
        results = {
            'model_name': 'DiffuVQA-PubMedBERT',
            'dataset_name': 'SLAKE',
            'export_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'overall_accuracy': correct_all / total_samples if total_samples > 0 else 0.0,
            'yes_no_accuracy': correct_yn / count_yn if count_yn > 0 else 0.0,
            'open_ended_accuracy': correct_oe / count_oe if count_oe > 0 else 0.0,
            'bleu_1_score': np.mean(bleu1_scores) if bleu1_scores else 0.0,
            'rouge_l_score': np.mean(rougeL_scores) if rougeL_scores else 0.0,
            'meteor_score': np.mean(meteor_scores) if meteor_scores else 0.0,
            'cider_score': cider,
            'bert_score': bert_f1_score,
            'f1_score': np.mean(f1_scores) if f1_scores else 0.0,
            'additional_info': f"Empty: {empty_count} ({100*empty_count/total_samples:.1f}%)" if total_samples > 0 else "",
            'Sample Folder': sample_folder,
            'Total Samples': total_samples,
            'Evaluation Date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'File Count': 1
        }
        
        all_results.append(results)
        
        # Sonuçları yazdır
        print(f"\n📈 Sonuçlar - {os.path.basename(sample_file)}:")
        print(f"  Total Samples: {results['Total Samples']}")
        print(f"  Overall Accuracy: {results['overall_accuracy']:.4f}")
        print(f"  Yes/No Accuracy: {results['yes_no_accuracy']:.4f}")
        print(f"  Open-Ended Accuracy: {results['open_ended_accuracy']:.4f}")
        print(f"  F1 Score: {results['f1_score']:.4f}")
        print(f"  BLEU-1: {results['bleu_1_score']:.4f}")
        print(f"  ROUGE-L: {results['rouge_l_score']:.4f}")
        print(f"  METEOR: {results['meteor_score']:.4f}")
        print(f"  CIDEr: {results['cider_score']:.4f}")
        print(f"  BERTScore: {results['bert_score']:.4f}")
    
    # DataFrame oluştur ve CSV'ye kaydet
    df = pd.DataFrame(all_results)
    df.to_csv(output_csv, index=False, encoding='utf-8')
    
    print(f"\n✅ Sonuçlar CSV'ye kaydedildi: {output_csv}")
    
    return df

print("✅ CSV export fonksiyonu hazır")

In [ ]:
# Tüm örnek dosyalarını değerlendir
sample_folder = "./samples/"
sample_files = glob.glob(f"{sample_folder}/*.jsonl")

# CSV çıktı dosyası (her durumda tanımla)
output_csv = "./reports/pubmedbert_evaluation_results.csv"
os.makedirs("./reports", exist_ok=True)

if not sample_files:
    print("⚠️ Örnek dosyası bulunamadı!")
    print("💡 Lütfen önce Cell 11'deki sampling (örnekleme) kodunu çalıştırın.")
    print(f"   Checkpoint dosyası: ./checkpoints/pubmedbert_slake/ema_0.9999_*.pt")
else:
    print(f"📂 {len(sample_files)} örnek dosyası bulundu")
    
    # Değerlendirme yap ve CSV'ye kaydet
    results_df = evaluate_and_export_csv(sample_files, output_csv=output_csv)
    
    # Sonuçları görüntüle
    print("\n" + "="*80)
    print("📊 TÜM SONUÇLAR:")
    print("="*80)
    display(results_df)

## 5. Sonuçları İndir

In [ ]:
# CSV dosyasını Google Drive'a kaydet ve indir
from google.colab import files

# CSV yolunu kontrol et (önceki hücrede tanımlanmış olmalı)
if 'output_csv' not in locals():
    output_csv = "./reports/pubmedbert_evaluation_results.csv"

# CSV'yi Google Drive'a kaydet
drive_output_path = '/content/drive/MyDrive/DiffuVQA_Results/'
os.makedirs(drive_output_path, exist_ok=True)

if os.path.exists(output_csv):
    # Drive'a kopyala
    import shutil
    shutil.copy(output_csv, drive_output_path)
    print(f"✅ CSV Google Drive'a kaydedildi: {drive_output_path}")
    
    # Lokal olarak da indir
    files.download(output_csv)
    print(f"✅ {output_csv} bilgisayarınıza indirildi")
else:
    print(f"⚠️ CSV dosyası bulunamadı: {output_csv}")
    print("💡 Lütfen önce değerlendirme hücresini (Cell 15) çalıştırın.")

# Checkpoint'leri de Drive'a yedekle (opsiyonel)
print("\n💾 Checkpoint'leri yedeklemek ister misiniz?")
print("   Aşağıdaki komutu uncomment edin:")
print("   # !cp -r ./checkpoints /content/drive/MyDrive/DiffuVQA_Checkpoints/")

import matplotlib.pyplot as plt
import seaborn as sns

# Sonuçları görselleştir
if 'results_df' in locals() and len(results_df) > 0:
    # Metrik sütunlarını seç
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Accuracy metrikleri
    accuracy_cols = ['Overall Accuracy', 'CLOSED Accuracy', 'OPEN Accuracy']
    results_df[accuracy_cols].iloc[0].plot(kind='bar', ax=axes[0], color='skyblue')
    axes[0].set_title('Accuracy Metrics (CLOSED=Yes/No, OPEN=Open-Ended)', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Score', fontsize=12)
    axes[0].set_ylim([0, 1])
    axes[0].grid(axis='y', alpha=0.3)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
    
    # NLG ve Semantic metrikleri
    nlg_cols = ['F1 Score', 'BLEU-1', 'ROUGE-L', 'METEOR', 'CIDEr', 'BERTScore F1']
    results_df[nlg_cols].iloc[0].plot(kind='bar', ax=axes[1], color='lightcoral')
    axes[1].set_title('NLG & Semantic Similarity Metrics', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Score', fontsize=12)
    axes[1].set_ylim([0, 1])
    axes[1].grid(axis='y', alpha=0.3)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.savefig('./reports/metrics_visualization.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Görselleştirme kaydedildi: ./reports/metrics_visualization.png")
else:
    print("⚠️ Görselleştirme için sonuç bulunamadı.")
    print("💡 Lütfen önce değerlendirme hücresini (Cell 15) çalıştırın.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sonuçları görselleştir
if 'results_df' in locals() and len(results_df) > 0:
    # Metrik sütunlarını seç
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Accuracy metrikleri (YENİ KOLONLAR: CLOSED ve OPEN)
    accuracy_cols = ['Overall Accuracy', 'CLOSED Accuracy', 'OPEN Accuracy']
    results_df[accuracy_cols].iloc[0].plot(kind='bar', ax=axes[0], color='skyblue')
    axes[0].set_title('Accuracy Metrics (CLOSED=Yes/No, OPEN=Open-Ended)', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Score', fontsize=12)
    axes[0].set_ylim([0, 1])
    axes[0].grid(axis='y', alpha=0.3)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
    
    # NLG ve Semantic metrikleri
    nlg_cols = ['F1 Score', 'BLEU-1', 'ROUGE-L', 'METEOR', 'CIDEr', 'BERTScore F1']
    results_df[nlg_cols].iloc[0].plot(kind='bar', ax=axes[1], color='lightcoral')
    axes[1].set_title('NLG & Semantic Similarity Metrics', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Score', fontsize=12)
    axes[1].set_ylim([0, 1])
    axes[1].grid(axis='y', alpha=0.3)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.savefig('./reports/metrics_visualization.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Görselleştirme kaydedildi: ./reports/metrics_visualization.png")
else:
    print("⚠️ Görselleştirme için sonuç bulunamadı.")
    print("💡 Lütfen önce değerlendirme hücresini (Cell 15) çalıştırın.")

---

## Notlar

- **PubMedBERT:** Medical domain'e özgü pre-trained embeddings kullanır
- **SLAKE Dataset:** Medical VQA için kullanılır
- **CSV Export:** Tüm metrikler otomatik olarak CSV'ye kaydedilir
- **Checkpoint:** Model checkpoint'leri `./checkpoints/` dizinine kaydedilir
- **Samples:** Model çıktıları `./samples/` dizinine kaydedilir

---

**Hazırlayan:** DiffuVQA Team  
**Tarih:** 8 Aralık 2025